In [1]:
import os
import glob
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

# Enforce reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

Compute Device: cuda


## 1. Evaluation Framework & Data Split Strategy

**Objective:** To accurately evaluate how well the drone gesture classifier will perform in a real-world deployment (e.g., the upcoming client demo).

**Methodology & Justification:** A standard randomized 80/20 train-validation split would result in *data leakage*. Because the dataset consists of specific subjects (e.g., S1, S3, S11) performing multiple iterations of gestures, randomly splitting instances would mean frames of the same subject appear in both the training and validation sets. In this scenario, the network might memorize subject-specific features (e.g., clothing color, room lighting, body proportions) rather than learning the generalized spatial geometry of the hand gestures. 

To ensure the model generalizes to *unseen humans*—a mandatory requirement for demo day—we implement a **Subject-Wise Split**. We isolate a subset of subjects completely from the training phase and reserve them exclusively for validation. This rigorous evaluation framework guarantees that our validation metrics reflect actual deployment readiness.

In [2]:
class DroneGestureDataset(Dataset):
    def __init__(self, root_dir, is_train=True, val_subjects=None, transform=None):
        """
        Args:
            root_dir (str): Path to the 'data_resized' directory.
            is_train (bool): If True, loads training subjects. If False, loads validation subjects.
            val_subjects (list): List of subject folder names to be used for validation (e.g., ['S13', 'S14', 'S15']).
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train
        
        if val_subjects is None:
            # Defaulting to 3 subjects for validation to test generalization
            val_subjects = ['S13', 'S14', 'S15'] 
            
        self.instances = [] 
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # Traverse the hierarchical directory structure
        for cls_name in self.classes:
            cls_path = os.path.join(root_dir, cls_name)
            
            for subject_id in os.listdir(cls_path):
                subj_path = os.path.join(cls_path, subject_id)
                if not os.path.isdir(subj_path): continue

                # Subject-wise split logic
                is_val_subject = subject_id in val_subjects
                if (self.is_train and is_val_subject) or (not self.is_train and not is_val_subject):
                    continue 

                for instance_id in os.listdir(subj_path):
                    inst_path = os.path.join(subj_path, instance_id)
                    if not os.path.isdir(inst_path): continue

                    # Grab all PNG frames and sort them to maintain temporal order (1 to 5)
                    frames = sorted(glob.glob(os.path.join(inst_path, '*.png')))
                    if len(frames) == 5:
                        self.instances.append((frames, self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        frame_paths, label = self.instances[idx]
        frames = []

        # Load and transform each frame in the instance sequence
        for path in frame_paths:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)

        # Stack the 5 frames. Resulting shape: [5, Channels, Height, Width]
        frames_tensor = torch.stack(frames)
        
        return frames_tensor, label

print("Dataset class successfully defined.")

Dataset class successfully defined.


In [3]:
# Cell 4: Data Transformations & Loaders

# Define the spatial dimensions required by standard pre-trained CNNs
img_height, img_width = 224, 224

# Training transforms include data augmentation to help the model generalize
train_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    # Standard ImageNet normalization values
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Validation transforms only resize and normalize, no augmentation
val_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Initialize the datasets using the custom class
root_data_path = './data_resized'

train_dataset = DroneGestureDataset(
    root_dir=root_data_path, 
    is_train=True, 
    transform=train_transforms
)

val_dataset = DroneGestureDataset(
    root_dir=root_data_path, 
    is_train=False, 
    transform=val_transforms
)

# Set up DataLoaders to batch and shuffle the data for training
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training instances: {len(train_dataset)}")
print(f"Validation instances: {len(val_dataset)}")

Training instances: 817
Validation instances: 242


## 2. Network Architecture: Late Fusion 2D CNN

**Objective:** To process a sequence of 5 frames per instance using only a 2-D Convolutional Neural Network, ensuring the model remains lightweight enough for an embedded drone system while still utilizing all available temporal information.

**Methodology & Justification:** Instead of modifying a 2D CNN to accept a 15-channel stacked input (Early Fusion)—which would destroy the pre-trained first-layer weights and risk overfitting on our small dataset—we implement a **Late Fusion (Feature Aggregation)** architecture. 

We utilize a pre-trained `ResNet18` as the feature extraction backbone. `ResNet18` is chosen because it offers an excellent balance between representational capacity and computational efficiency, making it highly suitable for hardware-constrained environments. 

During the forward pass, the 5 frames of an instance are passed through the 2D CNN independently. The network extracts 5 distinct feature vectors. We then apply Mean Pooling across the temporal dimension to aggregate these 5 vectors into a single robust representation of the gesture. Finally, this aggregated vector is passed to a fully connected linear layer for the 13-class prediction. This approach preserves 100% of the ImageNet pre-trained weights, efficiently utilizes all 5 frames, and strictly adheres to the hardware and architectural constraints.

In [4]:
# Cell 6: Late Fusion Network Architecture

class LateFusionGestureModel(nn.Module):
    def __init__(self, num_classes=13):
        super(LateFusionGestureModel, self).__init__()
        
        # Load a pre-trained ResNet18 backbone for feature extraction
        # Weights are downloaded automatically from torchvision
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Extract the number of input features going into the original classifier
        in_features = self.backbone.fc.in_features
        
        # Remove the final fully connected layer so the backbone only outputs feature vectors
        self.backbone.fc = nn.Identity()
        
        # Define our custom classification head for the 13 gesture classes
        self.classifier = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        # x input shape: [batch_size, num_frames, channels, height, width]
        batch_size, num_frames, c, h, w = x.size()
        
        # Merge the batch and frame dimensions so the 2D CNN can process all images
        # New shape: [batch_size * 5, 3, 224, 224]
        x = x.view(batch_size * num_frames, c, h, w)
        
        # Pass the images through the backbone to get feature vectors
        # Output shape: [batch_size * 5, 512]
        features = self.backbone(x)
        
        # Separate the batch and frame dimensions again
        # New shape: [batch_size, 5, 512]
        features = features.view(batch_size, num_frames, -1)
        
        # Aggregate the features across the 5 frames using Mean Pooling
        # This condenses the sequence into a single representative vector per instance
        # Output shape: [batch_size, 512]
        aggregated_features = torch.mean(features, dim=1)
        
        # Pass the final aggregated vector through the classification head
        # Output shape: [batch_size, 13]
        out = self.classifier(aggregated_features)
        
        return out

# Instantiate the model and move it to the configured device (CPU or GPU)
model = LateFusionGestureModel(num_classes=13).to(device)
print("Late Fusion Model successfully initialized.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/ec2-user/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 258MB/s]


Late Fusion Model successfully initialized.
